# W2 Day3（周三）：损失函数 + 评估指标 + 数据增强

**日期**: 04/16 | **时长**: 3h | **产出物**: 指标速查卡

---

## 🗺️ 今日路线图

| 模块 | 内容 | 时长 |
|------|------|------|
| 1 | Cross-Entropy Loss 深度剖析 | 20min |
| 2 | Focal Loss：类别不平衡杀手 | 20min |
| 3 | Triplet Loss + InfoNCE：度量学习 | 25min |
| 4 | Precision / Recall / F1 完整实验 | 25min |
| 5 | mAP：目标检测核心指标 | 20min |
| 6 | FID + CLIP Score：生成模型指标 | 25min |
| 7 | 数据增强消融实验 | 25min |
| 8 | 过拟合诊断 + 正则化实验 | 20min |
| 9 | 指标速查卡 + 总结 | 10min |

**前置依赖**: W1 Day3-4 (PyTorch基础), W1 Day6 (ResNet/ViT), W2 Day2 (CLIP/InfoNCE)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import time

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
torch.manual_seed(42)
np.random.seed(42)

---
## 模块 1：Cross-Entropy Loss 深度剖析 (20min)

### 1.1 从信息论到 CE Loss

**信息量**: 事件 $x$ 发生的信息量 $I(x) = -\log P(x)$

**熵**: 信息量的期望 $H(P) = -\sum_i P(x_i) \log P(x_i)$ — 衡量"不确定性"

**交叉熵**: $H(P, Q) = -\sum_i P(x_i) \log Q(x_i)$ — 用分布 $Q$ 编码分布 $P$ 的平均信息量

**KL 散度**: $D_{KL}(P \| Q) = H(P, Q) - H(P)$ — 交叉熵 - 熵

**分类任务中**: $P$ 是 one-hot 标签（熵=0），所以最小化 CE = 最小化 KL 散度

$$\mathcal{L}_{CE} = -\sum_{c=1}^{C} y_c \log(\hat{p}_c) = -\log(\hat{p}_{\text{target}})$$

In [ ]:
# ============================================================
# 实验 1.1: 手写 Cross-Entropy + 与 PyTorch 对比
# ============================================================

def cross_entropy_manual(logits, targets):
    """
    手写 CE Loss (数值稳定版)
    logits: (N, C) 未经 softmax 的原始输出
    targets: (N,) 类别索引
    
    关键: 不要先 softmax 再 log，而是用 log-sum-exp 技巧
    log(softmax(x_i)) = x_i - log(sum(exp(x_j)))
                      = x_i - max(x) - log(sum(exp(x_j - max(x))))  # 数值稳定
    """
    N, C = logits.shape
    
    # Log-Sum-Exp trick
    max_logits = logits.max(dim=1, keepdim=True).values  # (N, 1)
    log_sum_exp = max_logits.squeeze() + torch.log(
        torch.exp(logits - max_logits).sum(dim=1)
    )  # (N,)
    
    # 取出 target 对应的 logit
    target_logits = logits[torch.arange(N), targets]  # (N,)
    
    # CE = -log(softmax(target)) = -(target_logit - log_sum_exp)
    loss = -target_logits + log_sum_exp
    
    return loss.mean()

# 验证
torch.manual_seed(42)
logits = torch.randn(8, 10, requires_grad=True)  # 8 个样本，10 个类
targets = torch.randint(0, 10, (8,))

loss_manual = cross_entropy_manual(logits, targets)
loss_pytorch = F.cross_entropy(logits, targets)

print(f"手写 CE Loss:   {loss_manual.item():.6f}")
print(f"PyTorch CE Loss: {loss_pytorch.item():.6f}")
print(f"差异: {abs(loss_manual.item() - loss_pytorch.item()):.2e}")
print(f"✅ 一致！" if abs(loss_manual.item() - loss_pytorch.item()) < 1e-5 else "❌")

In [ ]:
# ============================================================
# 实验 1.2: 数值稳定性 — 为什么不能先 softmax 再 log
# ============================================================

def ce_naive_unstable(logits, targets):
    """不稳定版本: 先 softmax 再 log"""
    probs = torch.softmax(logits, dim=1)  # 可能下溢
    log_probs = torch.log(probs)           # log(0) = -inf !
    N = logits.shape[0]
    return -log_probs[torch.arange(N), targets].mean()

# 正常范围的 logits
logits_normal = torch.randn(4, 5)
targets_test = torch.tensor([0, 1, 2, 3])

print("正常范围 logits:")
print(f"  不稳定版: {ce_naive_unstable(logits_normal, targets_test):.6f}")
print(f"  稳定版:   {cross_entropy_manual(logits_normal, targets_test):.6f}")

# 极端 logits (大值)
logits_extreme = torch.tensor([[100., 0., 0., 0., 0.],
                                [0., 0., 0., 0., 100.],
                                [-100., 0., 0., 0., 0.],
                                [0., 0., 0., -100., 0.]])

print("\n极端 logits (含 ±100):")
print(f"  不稳定版: {ce_naive_unstable(logits_extreme, targets_test)}")
print(f"  稳定版:   {cross_entropy_manual(logits_extreme, targets_test):.6f}")
print(f"  PyTorch:  {F.cross_entropy(logits_extreme, targets_test):.6f}")

print("\n💡 PyTorch 的 F.cross_entropy 内部使用 log-sum-exp 技巧")
print("   永远不要自己先 softmax 再 log！直接传 logits 给 CE loss")

In [ ]:
# ============================================================
# 实验 1.3: CE Loss 的梯度直觉
# ============================================================

# CE 对 logits 的梯度 = softmax(logits) - one_hot(target)
# 即: 梯度 = 预测概率 - 真实标签

logits = torch.tensor([[2.0, 1.0, 0.5]], requires_grad=True)
target = torch.tensor([0])  # 正确类别是 0

loss = F.cross_entropy(logits, target)
loss.backward()

probs = torch.softmax(logits, dim=1).detach()
one_hot = F.one_hot(target, num_classes=3).float()
expected_grad = (probs - one_hot) / logits.shape[0]  # mean reduction

print("CE Loss 梯度解析:")
print(f"  Logits:       {logits.data.numpy()[0]}")
print(f"  Softmax:      {probs.numpy()[0]}")
print(f"  One-hot:      {one_hot.numpy()[0]}")
print(f"  预期梯度:     {expected_grad.numpy()[0]}")
print(f"  实际梯度:     {logits.grad.numpy()[0]}")
print(f"  ✅ 一致！" if torch.allclose(logits.grad, expected_grad, atol=1e-5) else "❌")

print("\n💡 梯度 = 预测概率 - 真实标签")
print("   → 预测越错（概率离 1 越远），梯度越大")
print("   → 预测正确时梯度接近 0（已经学好了）")
print("   这个性质非常直观、非常优美！")

In [ ]:
# ============================================================
# 实验 1.4: Label Smoothing — 防过拟合的正则化技巧
# ============================================================

def ce_label_smoothing(logits, targets, smoothing=0.1):
    """
    Label Smoothing CE Loss
    
    将 one-hot [1, 0, 0, ...] 变为 [1-ε+ε/C, ε/C, ε/C, ...]
    即: 真实类概率从 1.0 降为 1-ε+ε/C，其他类从 0 升为 ε/C
    """
    C = logits.shape[1]
    log_probs = F.log_softmax(logits, dim=1)
    
    # 构造 smooth 标签
    with torch.no_grad():
        smooth_labels = torch.full_like(log_probs, smoothing / C)
        smooth_labels.scatter_(1, targets.unsqueeze(1), 1.0 - smoothing + smoothing / C)
    
    # CE with smooth labels
    loss = -(smooth_labels * log_probs).sum(dim=1).mean()
    return loss

logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))

loss_standard = F.cross_entropy(logits, targets)
loss_smooth_01 = ce_label_smoothing(logits, targets, smoothing=0.1)
loss_smooth_02 = ce_label_smoothing(logits, targets, smoothing=0.2)
loss_pytorch_ls = F.cross_entropy(logits, targets, label_smoothing=0.1)

print(f"标准 CE:            {loss_standard.item():.6f}")
print(f"Label Smooth ε=0.1: {loss_smooth_01.item():.6f}")
print(f"Label Smooth ε=0.2: {loss_smooth_02.item():.6f}")
print(f"PyTorch LS ε=0.1:   {loss_pytorch_ls.item():.6f}")
print(f"手写 vs PyTorch LS:  {abs(loss_smooth_01.item() - loss_pytorch_ls.item()):.2e}")

print("\n💡 Label Smoothing 效果:")
print("  • 防止模型对预测过于自信 (logits 不会太极端)")
print("  • 相当于给 CE 加了一个 KL(uniform || softmax) 正则项")
print("  • 常用 ε=0.1，在 ImageNet 上能提升 ~0.2% accuracy")

---
## 模块 2：Focal Loss — 类别不平衡杀手 (20min)

### 2.1 问题：类别不平衡

在目标检测中，负样本(背景)远多于正样本(目标)，比例可达 1000:1。

标准 CE 对"简单负样本"也产生梯度 → 这些简单样本的梯度淹没了困难样本。

### 2.2 Focal Loss

$$\mathcal{L}_{FL} = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

- $p_t$ = 正确类别的预测概率
- $(1 - p_t)^\gamma$ 是 **调制因子**: 预测越准确 → 因子越小 → loss 越小
- $\gamma$ 通常取 2
- $\alpha_t$ 平衡类别权重

In [ ]:
# ============================================================
# 实验 2.1: 手写 Focal Loss
# ============================================================

class FocalLoss(nn.Module):
    """
    Focal Loss: 自动降低简单样本的权重
    
    当 γ=0: Focal Loss = CE Loss (退化为标准 CE)
    当 γ>0: 简单样本 (p_t 大) 的 loss 被大幅降低
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha  # 类别权重, (C,) tensor
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, logits, targets):
        # 计算标准 CE (不做 reduction)
        ce_loss = F.cross_entropy(logits, targets, reduction='none')  # (N,)
        
        # 计算 p_t (正确类别的预测概率)
        p_t = torch.exp(-ce_loss)  # p_t = exp(-CE) = softmax(logit_target)
        
        # 调制因子
        focal_weight = (1 - p_t) ** self.gamma
        
        # Focal Loss
        focal_loss = focal_weight * ce_loss
        
        # Alpha 权重
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

# 验证: gamma=0 时 = CE
logits = torch.randn(8, 10)
targets = torch.randint(0, 10, (8,))

ce = F.cross_entropy(logits, targets)
fl_g0 = FocalLoss(gamma=0.0)(logits, targets)
fl_g2 = FocalLoss(gamma=2.0)(logits, targets)

print(f"CE Loss:              {ce.item():.6f}")
print(f"Focal Loss (γ=0):     {fl_g0.item():.6f}  ← 应该等于 CE")
print(f"Focal Loss (γ=2):     {fl_g2.item():.6f}  ← 应该更小")
print(f"✅ γ=0 退化为 CE" if abs(ce.item() - fl_g0.item()) < 1e-5 else "❌")

In [ ]:
# ============================================================
# 实验 2.2: 可视化 Focal Loss 对不同难度样本的调节作用
# ============================================================

p_t = np.linspace(0.01, 0.99, 200)  # 正确类别的预测概率
ce_curve = -np.log(p_t)  # 标准 CE

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 不同 gamma 的 Focal Loss 曲线
gammas = [0, 0.5, 1, 2, 5]
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(gammas)))

for gamma, color in zip(gammas, colors):
    fl_curve = ((1 - p_t) ** gamma) * (-np.log(p_t))
    axes[0].plot(p_t, fl_curve, color=color, linewidth=2, label=f'γ={gamma}')

axes[0].set_xlabel('Probability of correct class (p_t)', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Focal Loss for different γ', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 5])

# 调制因子 (1-p_t)^gamma
for gamma, color in zip(gammas[1:], colors[1:]):
    weight = (1 - p_t) ** gamma
    axes[1].plot(p_t, weight, color=color, linewidth=2, label=f'γ={gamma}')

axes[1].set_xlabel('Probability of correct class (p_t)', fontsize=12)
axes[1].set_ylabel('Modulating factor (1-p_t)^γ', fontsize=12)
axes[1].set_title('How much loss is down-weighted', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# 标注关键区域
axes[1].axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].text(0.15, 0.85, '← Hard\nsamples', transform=axes[1].transAxes, fontsize=11,
             color='red', fontweight='bold')
axes[1].text(0.7, 0.15, 'Easy\nsamples →', transform=axes[1].transAxes, fontsize=11,
             color='green', fontweight='bold')

plt.tight_layout()
plt.show()

# 量化分析
print("\n量化: γ=2 时各难度样本的 loss 缩放")
print("-" * 50)
for p in [0.1, 0.3, 0.5, 0.7, 0.9, 0.99]:
    ce = -np.log(p)
    fl = ((1-p)**2) * ce
    ratio = fl / ce
    difficulty = "困难" if p < 0.3 else ("中等" if p < 0.7 else "简单")
    print(f"  p_t={p:.2f} ({difficulty}): CE={ce:.3f}, FL={fl:.3f}, 比例={ratio:.3f} ({ratio*100:.1f}%)")

print("\n💡 γ=2 时, p=0.9 的简单样本 loss 缩小到原来的 1%")
print("   而 p=0.1 的困难样本 loss 只缩小到 81%")
print("   → 模型自动聚焦在困难样本上！")

In [ ]:
# ============================================================
# 实验 2.3: 不平衡数据上 CE vs Focal Loss 对比
# ============================================================

# 构造不平衡数据集 (CIFAR-10, 类0只保留100个样本, 其他保留5000个)
full_train = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
)

# 构造不平衡索引
rare_class = 0  # airplane 作为稀有类
rare_count = 100
normal_count = 5000

indices = []
class_counts = Counter()
for idx in range(len(full_train)):
    _, label = full_train[idx]
    if label == rare_class and class_counts[label] < rare_count:
        indices.append(idx)
        class_counts[label] += 1
    elif label != rare_class and class_counts[label] < normal_count:
        indices.append(idx)
        class_counts[label] += 1

imbalanced_dataset = Subset(full_train, indices)
imbalanced_loader = DataLoader(imbalanced_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

print(f"不平衡数据集类别分布:")
for cls in range(10):
    count = class_counts[cls]
    bar = '█' * (count // 200)
    marker = " ← RARE" if cls == rare_class else ""
    print(f"  Class {cls}: {count:5d} {bar}{marker}")

In [ ]:
# 快速训练函数
def quick_train(model, train_loader, test_loader, criterion, device,
                epochs=15, lr=0.01):
    """快速训练并返回稀有类和总体的测试结果"""
    model = model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    for epoch in range(1, epochs + 1):
        model.train()
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            loss = criterion(model(data), target)
            loss.backward()
            optimizer.step()
        scheduler.step()
    
    # 评估
    model.eval()
    class_correct = torch.zeros(10)
    class_total = torch.zeros(10)
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            pred = model(data).argmax(dim=1)
            for c in range(10):
                mask = (target == c)
                class_correct[c] += (pred[mask] == c).sum().item()
                class_total[c] += mask.sum().item()
    
    class_acc = class_correct / class_total * 100
    overall_acc = class_correct.sum() / class_total.sum() * 100
    return class_acc, overall_acc.item()

# 简单的 CNN 模型
def make_simple_cnn():
    return nn.Sequential(
        nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
        nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
        nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(128, 10)
    )

# 训练对比
print("训练中... (CE vs Focal Loss on imbalanced data)")
print("这可能需要几分钟...\n")

# 1. 标准 CE
torch.manual_seed(42)
model_ce = make_simple_cnn()
ce_class_acc, ce_overall = quick_train(
    model_ce, imbalanced_loader, test_loader,
    nn.CrossEntropyLoss(), device, epochs=15
)

# 2. Focal Loss (γ=2)
torch.manual_seed(42)
model_fl = make_simple_cnn()
fl_class_acc, fl_overall = quick_train(
    model_fl, imbalanced_loader, test_loader,
    FocalLoss(gamma=2.0), device, epochs=15
)

# 结果对比
classes = ('airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print(f"{'Class':12s} {'CE':>8s} {'Focal':>8s} {'Δ':>8s}")
print("-" * 38)
for i in range(10):
    marker = " ★" if i == rare_class else ""
    delta = fl_class_acc[i] - ce_class_acc[i]
    print(f"{classes[i]:12s} {ce_class_acc[i]:7.1f}% {fl_class_acc[i]:7.1f}% {delta:+7.1f}%{marker}")

print("-" * 38)
print(f"{'Overall':12s} {ce_overall:7.1f}% {fl_overall:7.1f}% {fl_overall-ce_overall:+7.1f}%")
print(f"\n💡 观察稀有类 (airplane ★) 的准确率变化")

---
## 模块 3：Triplet Loss + InfoNCE — 度量学习 (25min)

### 3.1 Triplet Loss

目标: 让 anchor 与 positive 的距离 < anchor 与 negative 的距离 (至少差 margin)

$$\mathcal{L}_{triplet} = \max(0, \|f(a) - f(p)\|^2 - \|f(a) - f(n)\|^2 + \text{margin})$$

### 3.2 InfoNCE (Day2 回顾)

$$\mathcal{L}_{InfoNCE} = -\log \frac{\exp(\text{sim}(z_i, z_j^+) / \tau)}{\sum_{k} \exp(\text{sim}(z_i, z_k) / \tau)}$$

InfoNCE 本质上是在 **N-1 个负样本中识别唯一正样本** 的分类问题。

In [ ]:
# ============================================================
# 实验 3.1: 手写 Triplet Loss + 可视化
# ============================================================

class TripletLoss(nn.Module):
    """Triplet Loss with online mining"""
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin
    
    def forward(self, anchor, positive, negative):
        """
        anchor, positive, negative: (N, D) 嵌入向量
        """
        d_pos = F.pairwise_distance(anchor, positive)   # (N,)
        d_neg = F.pairwise_distance(anchor, negative)   # (N,)
        loss = F.relu(d_pos - d_neg + self.margin)      # (N,)
        return loss.mean()

# 创建 2D 嵌入演示
torch.manual_seed(42)
embed_dim = 2

# 三个类别的 "嵌入" (初始状态: 混在一起)
class_centers_init = torch.tensor([[0., 0.], [0.5, 0.5], [-0.5, 0.5]])
n_per_class = 30

# 模拟训练过程: 用 Triplet Loss 将类别分开
embeddings = nn.Parameter(
    torch.cat([
        class_centers_init[i].unsqueeze(0) + torch.randn(n_per_class, embed_dim) * 0.5
        for i in range(3)
    ])
)
labels = torch.cat([torch.full((n_per_class,), i) for i in range(3)])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
triplet_loss = TripletLoss(margin=1.0)
optimizer = optim.SGD([embeddings], lr=0.1)

colors = ['#e74c3c', '#3498db', '#2ecc71']

for step_idx, (ax, title, n_steps) in enumerate([
    (axes[0], 'Before Training', 0),
    (axes[1], 'After 50 Steps', 50),
    (axes[2], 'After 200 Steps', 150),  # 累积: 50+150=200
]):
    # 训练
    for _ in range(n_steps):
        optimizer.zero_grad()
        # 随机采样 triplets
        batch_loss = 0
        for _ in range(16):
            # 随机选 anchor
            a_idx = np.random.randint(len(labels))
            a_label = labels[a_idx].item()
            # 选 positive (同类)
            pos_indices = (labels == a_label).nonzero().squeeze()
            p_idx = pos_indices[np.random.randint(len(pos_indices))].item()
            # 选 negative (不同类)
            neg_indices = (labels != a_label).nonzero().squeeze()
            n_idx = neg_indices[np.random.randint(len(neg_indices))].item()
            
            batch_loss += triplet_loss(
                embeddings[a_idx:a_idx+1],
                embeddings[p_idx:p_idx+1],
                embeddings[n_idx:n_idx+1]
            )
        (batch_loss / 16).backward()
        optimizer.step()
    
    # 可视化
    emb = embeddings.detach().numpy()
    for i in range(3):
        mask = labels.numpy() == i
        ax.scatter(emb[mask, 0], emb[mask, 1], c=colors[i], s=40,
                   label=f'Class {i}', alpha=0.7, edgecolors='white', linewidth=0.5)
    ax.set_title(title, fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.suptitle('Triplet Loss: Pulling Same-class Together, Pushing Different-class Apart', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 实验 3.2: Triplet Loss vs InfoNCE 对比
# ============================================================

def info_nce_loss(features, labels, temperature=0.1):
    """
    InfoNCE: batch 内所有同类为正对，不同类为负对
    features: (N, D)
    labels: (N,)
    """
    features = F.normalize(features, dim=1)
    sim = features @ features.T / temperature  # (N, N)
    
    # 正对 mask: 同类且不是自己
    labels_eq = labels.unsqueeze(0) == labels.unsqueeze(1)  # (N, N)
    self_mask = ~torch.eye(len(labels), dtype=torch.bool)
    pos_mask = labels_eq & self_mask
    
    # 分母: 所有非自己的样本
    neg_mask = self_mask
    
    # 对每个 anchor, 计算 InfoNCE
    losses = []
    for i in range(len(labels)):
        pos_sim = sim[i][pos_mask[i]]  # 所有正对的相似度
        all_sim = sim[i][neg_mask[i]]  # 所有非自己的相似度
        if len(pos_sim) == 0:
            continue
        # 对每个正对, 计算 -log(exp(pos) / sum(exp(all)))
        for p in pos_sim:
            loss = -p + torch.logsumexp(all_sim, dim=0)
            losses.append(loss)
    
    return torch.stack(losses).mean()

# 对比三种 loss 在 embedding 质量上的差异
print("=" * 60)
print("损失函数对比: CE vs Triplet vs InfoNCE")
print("=" * 60)

comparison = [
    ("Cross-Entropy", "分类", "需要标签", "简单直接", "不直接优化嵌入距离"),
    ("Triplet Loss", "度量学习", "需要三元组", "直观的距离优化", "挖掘困难负样本很关键"),
    ("InfoNCE", "对比学习", "需要正负对", "batch 内高效", "对 batch size 敏感"),
]

print(f"{'Loss':15s} {'范式':8s} {'数据需求':12s} {'优势':15s} {'局限':20s}")
print("-" * 75)
for name, paradigm, data_req, advantage, limitation in comparison:
    print(f"{name:15s} {paradigm:8s} {data_req:12s} {advantage:15s} {limitation:20s}")

print("\n💡 在 CLIP/SimCLR 等框架中, InfoNCE 是首选:")
print("  • batch 内自动构造负对 (不需要显式挖掘)")
print("  • 大 batch size → 更多负样本 → 更好的效果")
print("  • 温度 τ 控制难度: 小 τ → 对困难负样本更敏感")

---
## 模块 4：Precision / Recall / F1 完整实验 (25min)

### 4.1 混淆矩阵基础

```
                    预测
              Positive  Negative
实际 Positive    TP        FN
     Negative    FP        TN
```

- **Precision** = TP / (TP + FP) — "预测为正的里面有多少真的是正的"
- **Recall** = TP / (TP + FN) — "真正为正的里面有多少被找出来了"
- **F1** = 2 × P × R / (P + R) — 调和平均

### 多分类拓展
- **Macro**: 每个类算 P/R/F1，再取平均（对小类更公平）
- **Micro**: 汇总所有 TP/FP/FN 再算（= Accuracy for single-label）
- **Weighted**: 按类别样本数加权

In [ ]:
# ============================================================
# 实验 4.1: 手写所有分类指标
# ============================================================

def compute_classification_metrics(y_true, y_pred, num_classes):
    """
    从零计算所有分类指标
    y_true, y_pred: (N,) 类别索引
    """
    # 混淆矩阵
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    
    # 每类指标
    per_class = {}
    for c in range(num_classes):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp  # 被预测为 c 但实际不是
        fn = cm[c, :].sum() - tp  # 实际是 c 但没被预测到
        tn = cm.sum() - tp - fp - fn
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        per_class[c] = {
            'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
            'precision': precision, 'recall': recall, 'f1': f1,
            'support': cm[c, :].sum()
        }
    
    # Macro 平均
    macro_p = np.mean([per_class[c]['precision'] for c in range(num_classes)])
    macro_r = np.mean([per_class[c]['recall'] for c in range(num_classes)])
    macro_f1 = np.mean([per_class[c]['f1'] for c in range(num_classes)])
    
    # Micro 平均 (= Accuracy for single-label)
    total_tp = sum(per_class[c]['tp'] for c in range(num_classes))
    total_fp = sum(per_class[c]['fp'] for c in range(num_classes))
    total_fn = sum(per_class[c]['fn'] for c in range(num_classes))
    micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0
    
    # Weighted 平均
    total_support = sum(per_class[c]['support'] for c in range(num_classes))
    weighted_f1 = sum(
        per_class[c]['f1'] * per_class[c]['support'] / total_support
        for c in range(num_classes)
    )
    
    accuracy = total_tp / len(y_true)
    
    return {
        'confusion_matrix': cm,
        'per_class': per_class,
        'accuracy': accuracy,
        'macro': {'precision': macro_p, 'recall': macro_r, 'f1': macro_f1},
        'micro': {'precision': micro_p, 'recall': micro_r, 'f1': micro_f1},
        'weighted_f1': weighted_f1,
    }

# 在训练好的模型上计算
model_ce.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for data, target in test_loader:
        data = data.to(device)
        pred = model_ce(data).argmax(dim=1).cpu()
        all_preds.extend(pred.numpy())
        all_labels.extend(target.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

metrics = compute_classification_metrics(all_labels, all_preds, 10)

# 打印报告
print("分类报告 (CE model on imbalanced data):")
print("=" * 65)
print(f"{'Class':12s} {'P':>7s} {'R':>7s} {'F1':>7s} {'Support':>8s}")
print("-" * 65)
for c in range(10):
    m = metrics['per_class'][c]
    marker = " ★" if c == rare_class else ""
    print(f"{classes[c]:12s} {m['precision']:7.3f} {m['recall']:7.3f} "
          f"{m['f1']:7.3f} {m['support']:8d}{marker}")
print("-" * 65)
print(f"{'Accuracy':12s} {'':>7s} {'':>7s} {metrics['accuracy']:7.3f} {len(all_labels):8d}")
print(f"{'Macro avg':12s} {metrics['macro']['precision']:7.3f} "
      f"{metrics['macro']['recall']:7.3f} {metrics['macro']['f1']:7.3f}")
print(f"{'Weighted avg':12s} {'':>7s} {'':>7s} {metrics['weighted_f1']:7.3f}")

print("\n💡 注意稀有类 airplane ★ 的 P/R/F1 通常较低")
print("   Macro F1 对类别不平衡更敏感 (每类等权)")

In [ ]:
# ============================================================
# 实验 4.2: 可视化混淆矩阵
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 原始混淆矩阵
cm = metrics['confusion_matrix']
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks(range(10))
axes[0].set_yticks(range(10))
axes[0].set_xticklabels(classes, rotation=45, ha='right', fontsize=9)
axes[0].set_yticklabels(classes, fontsize=9)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix (counts)')
plt.colorbar(im, ax=axes[0])

# 添加数字标注
for i in range(10):
    for j in range(10):
        color = 'white' if cm[i,j] > cm.max()/2 else 'black'
        axes[0].text(j, i, str(cm[i,j]), ha='center', va='center',
                     fontsize=7, color=color)

# 归一化混淆矩阵 (每行除以该行总和)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
im2 = axes[1].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
axes[1].set_xticks(range(10))
axes[1].set_yticks(range(10))
axes[1].set_xticklabels(classes, rotation=45, ha='right', fontsize=9)
axes[1].set_yticklabels(classes, fontsize=9)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].set_title('Normalized Confusion Matrix (= Recall per class)')
plt.colorbar(im2, ax=axes[1])

for i in range(10):
    for j in range(10):
        color = 'white' if cm_norm[i,j] > 0.5 else 'black'
        axes[1].text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                     fontsize=7, color=color)

plt.tight_layout()
plt.show()

print("💡 对角线值 = 各类 Recall")
print("   非对角线 = 混淆: 最常见的是 cat↔dog, deer↔horse")

---
## 模块 5：mAP — 目标检测核心指标 (20min)

### 5.1 AP (Average Precision) 的计算

1. 按 confidence score 排序所有检测结果
2. 逐个计算 Precision-Recall 曲线上的点
3. AP = PR 曲线下面积 (使用 all-point interpolation)

### 5.2 mAP = 所有类别 AP 的平均

In [ ]:
# ============================================================
# 实验 5.1: 从零计算 AP + 绘制 PR 曲线
# ============================================================

def compute_ap(y_true, scores, num_recall_points=11):
    """
    计算 Average Precision
    y_true: (N,) bool, 是否为正样本
    scores: (N,) float, 置信度分数
    
    返回: AP, precisions, recalls
    """
    # 按 score 降序排列
    sorted_indices = np.argsort(-scores)
    y_true_sorted = y_true[sorted_indices]
    
    # 逐步计算 P 和 R
    tp_cumsum = np.cumsum(y_true_sorted)
    fp_cumsum = np.cumsum(~y_true_sorted)
    
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum)
    recalls = tp_cumsum / y_true.sum()
    
    # 在开头加 (recall=0, precision=1) 的点
    precisions = np.concatenate([[1.0], precisions])
    recalls = np.concatenate([[0.0], recalls])
    
    # All-point interpolation: 对每个 recall 值，取该 recall 及以后的最大 precision
    # (从右往左取累积最大值)
    precisions_interp = np.maximum.accumulate(precisions[::-1])[::-1]
    
    # AP = 曲线下面积 (梯形法)
    ap = 0.0
    for i in range(1, len(recalls)):
        ap += (recalls[i] - recalls[i-1]) * precisions_interp[i]
    
    return ap, precisions, recalls, precisions_interp

# 模拟目标检测结果
np.random.seed(42)
n_detections = 100
n_positives = 20  # 实际有 20 个目标

# 分数: 正样本的分数平均更高但有重叠
is_positive = np.array([True]*n_positives + [False]*(n_detections-n_positives))
scores = np.where(is_positive, 
                  np.random.beta(5, 2, n_detections),  # 正样本: 偏高分
                  np.random.beta(2, 5, n_detections))  # 负样本: 偏低分

# 打乱
perm = np.random.permutation(n_detections)
is_positive = is_positive[perm]
scores = scores[perm]

ap, prec, rec, prec_interp = compute_ap(is_positive, scores)

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR 曲线
axes[0].step(rec, prec, 'b-', alpha=0.5, where='post', label='Raw PR curve')
axes[0].step(rec, prec_interp, 'r-', where='post', linewidth=2, 
              label=f'Interpolated (AP={ap:.3f})')
axes[0].fill_between(rec, prec_interp, step='post', alpha=0.1, color='red')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title(f'Precision-Recall Curve (AP = {ap:.3f})')
axes[0].legend(fontsize=11)
axes[0].set_xlim([0, 1.05])
axes[0].set_ylim([0, 1.05])
axes[0].grid(True, alpha=0.3)

# Score 分布
axes[1].hist(scores[is_positive], bins=20, alpha=0.6, label='Positive', color='green')
axes[1].hist(scores[~is_positive], bins=20, alpha=0.6, label='Negative', color='red')
axes[1].set_xlabel('Confidence Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Score Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"AP = {ap:.3f}")
print(f"\n💡 mAP@0.5 (COCO): IoU 阈值 0.5 下的 mAP")
print(f"   mAP@0.5:0.95 (COCO): IoU 从 0.5 到 0.95 步长 0.05 取平均")
print(f"   COCO 主指标是 mAP@0.5:0.95, 比 mAP@0.5 严格得多")

---
## 模块 6：FID + CLIP Score — 生成模型指标 (25min)

### 6.1 FID (Fréchet Inception Distance)

衡量生成图像分布与真实图像分布的距离：

$$\text{FID} = \|\mu_r - \mu_g\|^2 + \text{Tr}(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2})$$

- 用 Inception V3 提取特征
- 假设特征服从高斯分布
- **越低越好** (0 = 完美)

### 6.2 CLIP Score

衡量图像与文本的对齐度：

$$\text{CLIP Score} = \max(100 \cdot \cos(E_I(img), E_T(text)), 0)$$

- **越高越好**

In [ ]:
# ============================================================
# 实验 6.1: 手写 FID 计算 (用简化的特征模拟)
# ============================================================

def compute_fid(mu1, sigma1, mu2, sigma2):
    """
    计算两个高斯分布之间的 Fréchet Distance
    mu1, mu2: (D,) 均值
    sigma1, sigma2: (D, D) 协方差矩阵
    """
    # 均值差的平方和
    diff = mu1 - mu2
    mean_diff_sq = np.dot(diff, diff)
    
    # 矩阵平方根: (sigma1 @ sigma2)^(1/2)
    # 用特征值分解计算
    product = sigma1 @ sigma2
    eigvals, eigvecs = np.linalg.eigh(product)
    # 数值稳定: 截断负的小特征值
    eigvals = np.maximum(eigvals, 0)
    sqrt_product = eigvecs @ np.diag(np.sqrt(eigvals)) @ eigvecs.T
    
    # FID
    trace_term = np.trace(sigma1 + sigma2 - 2 * sqrt_product)
    fid = mean_diff_sq + trace_term
    
    return fid

# 模拟实验: 不同质量的 "生成" 分布
np.random.seed(42)
D = 64  # 特征维度

# 真实分布
mu_real = np.random.randn(D)
A = np.random.randn(D, D) * 0.5
sigma_real = A @ A.T + np.eye(D) * 0.1

scenarios = {
    '完美生成 (same dist)': (mu_real, sigma_real),
    '轻微偏移': (mu_real + np.random.randn(D) * 0.5, sigma_real),
    '中等偏移': (mu_real + np.random.randn(D) * 2.0, sigma_real),
    '严重偏移': (mu_real + np.random.randn(D) * 5.0, sigma_real),
    '模式坍缩 (low var)': (mu_real, sigma_real * 0.1),
    '完全随机': (np.random.randn(D) * 3, np.eye(D) * 3),
}

print("FID 实验: 不同生成质量的 FID 值")
print("=" * 55)
fid_values = []
names = []
for name, (mu_gen, sigma_gen) in scenarios.items():
    fid = compute_fid(mu_real, sigma_real, mu_gen, sigma_gen)
    fid_values.append(fid)
    names.append(name)
    quality = "🟢" if fid < 10 else ("🟡" if fid < 100 else "🔴")
    print(f"  {quality} {name:30s}: FID = {fid:8.1f}")

print("\n💡 FID 参考值 (ImageNet 256×256):")
print("   DDPM: ~11, LDM: ~3.6, DiT-XL: ~2.3")
print("   StyleGAN3: ~2.8, Consistency Model: ~2.0")

In [ ]:
# ============================================================
# 实验 6.2: 手写 CLIP Score (使用模拟嵌入)
# ============================================================

def compute_clip_score(image_embeds, text_embeds):
    """
    CLIP Score = max(100 * cos_sim(image, text), 0)
    image_embeds: (N, D)
    text_embeds: (N, D)
    """
    # L2 归一化
    image_embeds = image_embeds / image_embeds.norm(dim=1, keepdim=True)
    text_embeds = text_embeds / text_embeds.norm(dim=1, keepdim=True)
    
    # 余弦相似度
    cos_sim = (image_embeds * text_embeds).sum(dim=1)  # (N,)
    
    # CLIP Score
    clip_score = torch.clamp(100 * cos_sim, min=0)
    
    return clip_score.mean().item(), clip_score

# 模拟不同对齐质量的图像-文本对
torch.manual_seed(42)
D = 512
N = 50

# 场景 1: 完美对齐 (图文嵌入相同方向)
base = F.normalize(torch.randn(N, D), dim=1)
noise_levels = [0.0, 0.1, 0.5, 1.0, 2.0]

print("CLIP Score 与对齐质量的关系:")
print("=" * 50)

clip_scores = []
for noise in noise_levels:
    text_embed = base + torch.randn(N, D) * noise
    score, _ = compute_clip_score(base, text_embed)
    clip_scores.append(score)
    quality = "🟢" if score > 25 else ("🟡" if score > 15 else "🔴")
    print(f"  {quality} Noise={noise:.1f}: CLIP Score = {score:.1f}")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(noise_levels, clip_scores, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Noise Level (misalignment)')
axes[0].set_ylabel('CLIP Score')
axes[0].set_title('CLIP Score vs Image-Text Alignment')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=25, color='green', linestyle='--', alpha=0.5, label='Good threshold')
axes[0].legend()

# FID vs CLIP Score 关系示意
axes[1].text(0.5, 0.5, 
    'FID vs CLIP Score\n\n'
    '• FID: 分布级别 (统计距离)\n'
    '  越低越好 | 需要大量样本\n\n'
    '• CLIP Score: 样本级别 (语义对齐)\n'
    '  越高越好 | 单张即可计算\n\n'
    '• 好的生成模型: 低 FID + 高 CLIP Score\n\n'
    '• Trade-off: 提高多样性(↓FID) 可能\n'
    '  降低文本对齐(↓CLIP Score)',
    transform=axes[1].transAxes, fontsize=12,
    verticalalignment='center', horizontalalignment='center',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
axes[1].axis('off')
axes[1].set_title('FID vs CLIP Score Summary')

plt.tight_layout()
plt.show()

---
## 模块 7：数据增强消融实验 (25min)

### 常见数据增强

| 增强 | 说明 | 常用场景 |
|------|------|----------|
| RandomCrop | 随机裁剪 | 几乎所有分类任务 |
| HorizontalFlip | 水平翻转 | 非文字/方向敏感任务 |
| ColorJitter | 颜色抖动 | 抗光照变化 |
| RandomErasing | 随机遮挡 | 防过拟合，类似 Cutout |
| MixUp | 线性混合两张图和标签 | 正则化 |
| CutMix | 裁剪贴合 | 比 MixUp 保留更多空间信息 |
| AutoAugment | 搜索最优增强策略 | 大规模训练 |

In [ ]:
# ============================================================
# 实验 7.1: 可视化各种数据增强
# ============================================================

# 获取一张原始图片
raw_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True,
    transform=transforms.ToTensor()
)
original_img, label = raw_dataset[0]

# 定义各种增强
augmentations = {
    'Original': transforms.Compose([]),
    'RandomCrop(32,4)': transforms.RandomCrop(32, padding=4),
    'HFlip': transforms.RandomHorizontalFlip(p=1.0),
    'ColorJitter': transforms.ColorJitter(brightness=0.4, contrast=0.4, 
                                           saturation=0.4, hue=0.1),
    'RandomRotation(15)': transforms.RandomRotation(15),
    'RandomErasing': transforms.Compose([
        transforms.RandomErasing(p=1.0, scale=(0.1, 0.3))
    ]),
    'GaussianBlur': transforms.GaussianBlur(kernel_size=3),
    'RandomAffine': transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for idx, (name, aug) in enumerate(augmentations.items()):
    ax = axes[idx // 4, idx % 4]
    if name == 'Original':
        img = original_img
    else:
        img = aug(original_img)
    
    # 显示
    ax.imshow(img.permute(1, 2, 0).numpy().clip(0, 1))
    ax.set_title(name, fontsize=11)
    ax.axis('off')

plt.suptitle(f'Data Augmentations on CIFAR-10 ({classes[label]})', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 实验 7.2: MixUp 和 CutMix 实现 + 可视化
# ============================================================

def mixup_data(x, y, alpha=0.2):
    """
    MixUp: x̃ = λx_i + (1-λ)x_j,  ỹ = λy_i + (1-λ)y_j
    lambda ~ Beta(alpha, alpha)
    """
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam

def cutmix_data(x, y, alpha=1.0):
    """
    CutMix: 从另一张图裁剪一块贴到当前图上
    标签按面积比例混合
    """
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size)
    
    _, _, H, W = x.shape
    
    # 随机裁剪区域
    cut_ratio = np.sqrt(1. - lam)
    cut_h = int(H * cut_ratio)
    cut_w = int(W * cut_ratio)
    
    cy = np.random.randint(H)
    cx = np.random.randint(W)
    
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    
    mixed_x = x.clone()
    mixed_x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
    
    # 实际的 lambda (按面积)
    lam_actual = 1 - (y2 - y1) * (x2 - x1) / (H * W)
    
    return mixed_x, y, y[index], lam_actual

# 可视化
batch = torch.stack([raw_dataset[i][0] for i in range(8)])
labels_batch = torch.tensor([raw_dataset[i][1] for i in range(8)])

fig, axes = plt.subplots(3, 4, figsize=(14, 10))

# 原图
for i in range(4):
    axes[0, i].imshow(batch[i].permute(1,2,0).numpy().clip(0,1))
    axes[0, i].set_title(f'{classes[labels_batch[i]]}', fontsize=11)
    axes[0, i].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=12)

# MixUp
np.random.seed(0)
mixed, y_a, y_b, lam = mixup_data(batch, labels_batch, alpha=0.4)
for i in range(4):
    axes[1, i].imshow(mixed[i].permute(1,2,0).numpy().clip(0,1))
    axes[1, i].set_title(f'λ={lam:.2f}: {classes[y_a[i]]}+{classes[y_b[i]]}', fontsize=9)
    axes[1, i].axis('off')
axes[1, 0].set_ylabel('MixUp', fontsize=12)

# CutMix
np.random.seed(0)
cutmixed, y_a, y_b, lam = cutmix_data(batch, labels_batch, alpha=1.0)
for i in range(4):
    axes[2, i].imshow(cutmixed[i].permute(1,2,0).numpy().clip(0,1))
    axes[2, i].set_title(f'λ={lam:.2f}: {classes[y_a[i]]}+{classes[y_b[i]]}', fontsize=9)
    axes[2, i].axis('off')
axes[2, 0].set_ylabel('CutMix', fontsize=12)

plt.suptitle('MixUp vs CutMix Data Augmentation', fontsize=14)
plt.tight_layout()
plt.show()

print("💡 MixUp: 全图像素级混合 → 图像变模糊，但标签精确")
print("   CutMix: 局部区域替换 → 图像保持清晰，空间信息更好")

In [ ]:
# ============================================================
# 实验 7.3: 数据增强消融实验
# ============================================================

# 使用完整 CIFAR-10 (平衡数据集)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

test_ds = torchvision.datasets.CIFAR10(root='./data', train=False, download=True,
                                        transform=test_transform)
test_ld = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2)

# 不同增强策略
aug_configs = {
    'No Aug': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]),
    '+Crop': transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]),
    '+Crop+Flip': transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]),
    '+Crop+Flip+Color': transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]),
    '+All+Erasing': transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        transforms.RandomErasing(p=0.5),
    ]),
}

ABLATION_EPOCHS = 15  # 减少 epoch 数加速
ablation_results = {}

print("数据增强消融实验 (CIFAR-10, 15 epochs each):")
print("=" * 55)

for name, transform in aug_configs.items():
    torch.manual_seed(42)
    train_ds = torchvision.datasets.CIFAR10(root='./data', train=True, download=True,
                                            transform=transform)
    train_ld = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)
    
    model = make_simple_cnn()
    _, overall = quick_train(model, train_ld, test_ld, nn.CrossEntropyLoss(),
                              device, epochs=ABLATION_EPOCHS, lr=0.05)
    ablation_results[name] = overall
    bar = '█' * int(overall / 2)
    print(f"  {name:25s}: {overall:.1f}% {bar}")

# 对比
print("\n增强策略增量分析:")
prev = None
for name, acc in ablation_results.items():
    delta = f"  ({acc - prev:+.1f}%)" if prev is not None else ""
    print(f"  {name:25s}: {acc:.1f}%{delta}")
    prev = acc

---
## 模块 8：过拟合诊断 + 正则化实验 (20min)

In [ ]:
# ============================================================
# 实验 8.1: 过拟合诊断 — Train vs Test Gap 分析
# ============================================================

def train_with_history(model, train_loader, test_loader, device, epochs=30, lr=0.05,
                        weight_decay=0., dropout=False):
    """训练并记录完整 train/test loss & acc 曲线"""
    model = model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}
    
    for epoch in range(epochs):
        # Train
        model.train()
        train_loss, correct, total = 0, 0, 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = F.cross_entropy(output, target)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * data.size(0)
            correct += output.argmax(1).eq(target).sum().item()
            total += data.size(0)
        scheduler.step()
        history['train_loss'].append(train_loss / total)
        history['train_acc'].append(100. * correct / total)
        
        # Test
        model.eval()
        test_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                test_loss += F.cross_entropy(output, target, reduction='sum').item()
                correct += output.argmax(1).eq(target).sum().item()
                total += data.size(0)
        history['test_loss'].append(test_loss / total)
        history['test_acc'].append(100. * correct / total)
    
    return history

# 用少量数据制造过拟合
small_train = Subset(raw_dataset, list(range(2000)))  # 只用 2000 张
small_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
small_train_ds = torchvision.datasets.CIFAR10(root='./data', train=True, download=True,
                                               transform=small_transform)
small_train_ds = Subset(small_train_ds, list(range(2000)))
small_train_loader = DataLoader(small_train_ds, batch_size=64, shuffle=True, num_workers=2)

print("训练中... (小数据集上观察过拟合)")
print(f"训练集大小: {len(small_train_ds)} (正常的 4%)")  

# 1. 无正则化 (预期严重过拟合)
torch.manual_seed(42)
hist_noreg = train_with_history(
    make_simple_cnn(), small_train_loader, test_ld, device,
    epochs=30, weight_decay=0.
)

# 2. Weight Decay
torch.manual_seed(42)
hist_wd = train_with_history(
    make_simple_cnn(), small_train_loader, test_ld, device,
    epochs=30, weight_decay=5e-3
)

print("训练完成！")

In [ ]:
# ============================================================
# 实验 8.2: 可视化过拟合诊断
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs_range = range(1, 31)

# 无正则化
axes[0,0].plot(epochs_range, hist_noreg['train_loss'], 'b-', label='Train', linewidth=2)
axes[0,0].plot(epochs_range, hist_noreg['test_loss'], 'r-', label='Test', linewidth=2)
axes[0,0].set_title('No Regularization — Loss', fontsize=13)
axes[0,0].set_xlabel('Epoch')
axes[0,0].set_ylabel('Loss')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# 标注过拟合开始的位置
test_losses = hist_noreg['test_loss']
min_idx = np.argmin(test_losses)
axes[0,0].axvline(x=min_idx+1, color='gray', linestyle='--', alpha=0.5)
axes[0,0].text(min_idx+2, max(test_losses)*0.9, f'过拟合\n开始 ≈ epoch {min_idx+1}',
               fontsize=10, color='red')

axes[0,1].plot(epochs_range, hist_noreg['train_acc'], 'b-', label='Train', linewidth=2)
axes[0,1].plot(epochs_range, hist_noreg['test_acc'], 'r-', label='Test', linewidth=2)
axes[0,1].set_title('No Regularization — Accuracy', fontsize=13)
axes[0,1].set_xlabel('Epoch')
axes[0,1].set_ylabel('Accuracy (%)')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

gap = [t - v for t, v in zip(hist_noreg['train_acc'], hist_noreg['test_acc'])]
axes[0,1].fill_between(epochs_range, hist_noreg['test_acc'], hist_noreg['train_acc'],
                        alpha=0.2, color='red', label=f'Gap (max={max(gap):.1f}%)')
axes[0,1].legend()

# Weight Decay
axes[1,0].plot(epochs_range, hist_wd['train_loss'], 'b-', label='Train', linewidth=2)
axes[1,0].plot(epochs_range, hist_wd['test_loss'], 'r-', label='Test', linewidth=2)
axes[1,0].set_title('With Weight Decay (5e-3) — Loss', fontsize=13)
axes[1,0].set_xlabel('Epoch')
axes[1,0].set_ylabel('Loss')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(epochs_range, hist_wd['train_acc'], 'b-', label='Train', linewidth=2)
axes[1,1].plot(epochs_range, hist_wd['test_acc'], 'r-', label='Test', linewidth=2)
axes[1,1].set_title('With Weight Decay — Accuracy', fontsize=13)
axes[1,1].set_xlabel('Epoch')
axes[1,1].set_ylabel('Accuracy (%)')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

gap_wd = [t - v for t, v in zip(hist_wd['train_acc'], hist_wd['test_acc'])]
axes[1,1].fill_between(epochs_range, hist_wd['test_acc'], hist_wd['train_acc'],
                        alpha=0.2, color='red', label=f'Gap (max={max(gap_wd):.1f}%)')
axes[1,1].legend()

plt.suptitle('Overfitting Diagnosis: No Reg vs Weight Decay (2K training samples)', fontsize=14)
plt.tight_layout()
plt.show()

print("\n📊 过拟合诊断清单:")
print(f"  无正则化: Train={hist_noreg['train_acc'][-1]:.1f}%, "
      f"Test={hist_noreg['test_acc'][-1]:.1f}%, Gap={gap[-1]:.1f}%")
print(f"  Weight Decay: Train={hist_wd['train_acc'][-1]:.1f}%, "
      f"Test={hist_wd['test_acc'][-1]:.1f}%, Gap={gap_wd[-1]:.1f}%")

print("\n💡 过拟合的信号:")
print("  1. Test loss 上升而 Train loss 继续下降")
print("  2. Train-Test accuracy gap 持续增大")
print("  3. 权重范数不断增大")
print("\n💡 解决方案 (按优先级):")
print("  1. 更多数据 (最有效) / 数据增强")
print("  2. Weight Decay / Dropout")
print("  3. Early Stopping")
print("  4. Label Smoothing")
print("  5. 减小模型容量 (最后考虑)")

---
## 模块 9：指标速查卡 + 总结 (10min)

In [ ]:
# ============================================================
# 指标速查卡
# ============================================================

cheat_sheet = """
╔══════════════════════════════════════════════════════════════════════════╗
║                      ML 指标与损失函数速查卡                            ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                        ║
║  ▎损失函数                                                              ║
║  ├─ Cross-Entropy:  分类标配, 梯度=softmax-onehot                       ║
║  ├─ Focal Loss:     类别不平衡, γ=2 降低简单样本权重                      ║
║  ├─ Label Smooth:   CE + KL(uniform), ε=0.1, 防过拟合                   ║
║  ├─ Triplet Loss:   度量学习, d(a,p) < d(a,n) - margin                 ║
║  └─ InfoNCE/NT-Xent: 对比学习, batch内正负对, τ控制难度                   ║
║                                                                        ║
║  ▎分类指标                                                              ║
║  ├─ Accuracy:   (TP+TN)/All, 不平衡数据时有误导性                        ║
║  ├─ Precision:  TP/(TP+FP), "预测阳性中真阳性比例"                        ║
║  ├─ Recall:     TP/(TP+FN), "真阳性中被找到的比例"                        ║
║  ├─ F1:         2PR/(P+R), P和R的调和平均                               ║
║  ├─ Macro avg:  各类取平均 (对小类公平)                                   ║
║  └─ Weighted:   按类别样本数加权                                         ║
║                                                                        ║
║  ▎检测指标                                                              ║
║  ├─ AP:         PR曲线面积, 用all-point interpolation                    ║
║  ├─ mAP@0.5:    IoU≥0.5 的 AP 平均                                     ║
║  └─ mAP@.5:.95: IoU从0.5到0.95步长0.05, COCO主指标                     ║
║                                                                        ║
║  ▎生成指标                                                              ║
║  ├─ FID:        分布距离 (Inception特征), 越低越好                       ║
║  ├─ CLIP Score:  图文对齐度, 越高越好                                    ║
║  └─ IS:          多样性+清晰度 (少用了), 越高越好                         ║
║                                                                        ║
║  ▎数据增强优先级                                                         ║
║  ├─ 必选: RandomCrop + HorizontalFlip                                  ║
║  ├─ 推荐: ColorJitter + RandomErasing                                  ║
║  └─ 进阶: MixUp / CutMix / AutoAugment                                ║
║                                                                        ║
║  ▎过拟合诊断                                                             ║
║  ├─ 信号: Test loss ↑ + Train loss ↓ + Gap 增大                        ║
║  └─ 处方: 更多数据 > 数据增强 > WD/Dropout > EarlyStop > 小模型         ║
║                                                                        ║
╚══════════════════════════════════════════════════════════════════════════╝
"""
print(cheat_sheet)

---
## 📝 Day3 总结

### 今日收获

**损失函数**:
- CE Loss: `F.cross_entropy` 内部用 log-sum-exp 保证数值稳定；梯度 = softmax - one_hot
- Focal Loss: $(1-p_t)^\gamma$ 调制因子，γ=2 时 p=0.9 的简单样本 loss 缩为 1%
- Label Smoothing: 将 one-hot 软化为 $(1-\epsilon, \epsilon/C, ...)$，常用 ε=0.1
- Triplet Loss: margin-based 度量学习，需要好的采样策略
- InfoNCE: batch 内自动构造正负对，CLIP/SimCLR 的核心

**评估指标**:
- P/R/F1: 不平衡数据下用 Macro F1 而非 Accuracy
- mAP: 目标检测核心指标，从 PR 曲线积分
- FID: 生成质量（分布距离），越低越好
- CLIP Score: 图文对齐度，越高越好

**实践洞察**:
- 数据增强: Crop+Flip 是基线，每加一个增强都有增量
- 过拟合诊断: 看 train-test gap，Weight Decay 是最常用的正则

### 面试要点
1. CE Loss 的梯度是什么？→ softmax - one_hot
2. Focal Loss 如何处理类别不平衡？→ $(1-p_t)^\gamma$ 下调简单样本权重
3. mAP 怎么算？→ 每类 AP (PR 曲线面积) → 取平均
4. FID 衡量什么？→ 生成分布与真实分布的 Fréchet 距离
5. 什么信号说明过拟合？→ Test loss ↑ + Train loss ↓

### 明日预告
- W2 Day4: nanoGPT — 完整 Decoder 架构 + KV Cache + 采样策略